# 02 GPS Transaction Cross-Reference

成员 B：时空分析负责人，主导 Q2。

本 notebook 复用成员 A 的交易中间层，结合原始 GPS 数据完成停车事件提取、车辆日轨迹统计、未分配车辆分析、交易异常复核，以及 Q2 所需的图表与结构化输出。

**数据契约**
- 优先读取 `data/processed/transactions_long.csv`、`anomaly_transactions.csv`、`cc_loyalty_matched.csv`。
- loyalty 仅有日期级精度，不能伪造分钟级验证。
- GPS 证据优先于模糊交易推断。
- 高置信 CC-loyalty 匹配不等于最终身份结论。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from vast_mc2.config import PROCESSED_DATA_DIR, RAW_DATA_DIR, REPORTS_DIR, FIGURES_DIR

sns.set_theme(style='whitegrid', context='talk')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
gps_raw = pd.read_csv(RAW_DATA_DIR / 'MC2' / 'gps.csv', encoding='cp1252')
assignments = pd.read_csv(RAW_DATA_DIR / 'MC2' / 'car-assignments.csv', encoding='cp1252')
transactions = pd.read_csv(PROCESSED_DATA_DIR / 'transactions_long.csv')
anomaly_transactions = pd.read_csv(PROCESSED_DATA_DIR / 'anomaly_transactions.csv')
cc_clean = pd.read_csv(PROCESSED_DATA_DIR / 'cc_clean.csv')
loyalty_clean = pd.read_csv(PROCESSED_DATA_DIR / 'loyalty_clean.csv')
matched = pd.read_csv(PROCESSED_DATA_DIR / 'cc_loyalty_matched.csv')
candidates = pd.read_csv(PROCESSED_DATA_DIR / 'cc_loyalty_match_candidates.csv')
location_category = pd.read_csv(PROCESSED_DATA_DIR / 'location_category.csv')

gps_raw['timestamp'] = pd.to_datetime(gps_raw['Timestamp'], format='%m/%d/%Y %H:%M:%S')
gps = gps_raw.rename(columns={'id':'vehicle_id', 'lat':'latitude', 'long':'longitude'}).sort_values(['vehicle_id', 'timestamp']).reset_index(drop=True)
gps.head()

## 1. GPS 轨迹解析与停车事件提取

停车事件的定义：同一车辆在相邻 GPS 点之间满足短时间间隔且位移很小，即可视作停留。这里采用的阈值为：

- 时间间隔 `<= 20s`
- 位移 `<= 35m`
- 停车段持续时间 `>= 1min`

该阈值对 MC2 的秒级 GPS 采样较稳健，可覆盖临停、门店停靠和短时驻留。

In [ ]:
gps['dt_s'] = gps.groupby('vehicle_id')['timestamp'].diff().dt.total_seconds()
gps['dlat'] = gps.groupby('vehicle_id')['latitude'].diff()
gps['dlon'] = gps.groupby('vehicle_id')['longitude'].diff()
gps['dist_m'] = np.sqrt((gps['dlat'] * 111000) ** 2 + (gps['dlon'] * 90000) ** 2)
gps['speed_mps'] = (gps['dist_m'] / gps['dt_s']).replace([np.inf, -np.inf], np.nan)
gps['date'] = gps['timestamp'].dt.date.astype('string')
gps['stationary'] = ((gps['dt_s'].fillna(0).between(0, 20)) & (gps['dist_m'].fillna(0) <= 35)) | gps['dt_s'].eq(0)
gps['state_change'] = gps.groupby('vehicle_id')['stationary'].transform(lambda s: (s != s.shift()).cumsum())
stationary = gps[gps['stationary']].copy()
stop_events = stationary.groupby(['vehicle_id', 'date', 'state_change']).agg(
    start_time=('timestamp', 'min'),
    end_time=('timestamp', 'max'),
    stop_points=('timestamp', 'size'),
    mean_lat=('latitude', 'mean'),
    mean_lon=('longitude', 'mean'),
    mean_speed_mps=('speed_mps', 'mean'),
).reset_index()
stop_events['duration_min'] = (stop_events['end_time'] - stop_events['start_time']).dt.total_seconds() / 60
stop_events = stop_events[stop_events['duration_min'] >= 1].reset_index(drop=True)
stop_events.to_csv(PROCESSED_DATA_DIR / 'gps_stop_events.csv', index=False)
stop_events.head()

In [ ]:
vehicle_daily = gps.groupby(['vehicle_id', 'date']).agg(
    points=('timestamp', 'size'),
    start_time=('timestamp', 'min'),
    end_time=('timestamp', 'max'),
    lat_min=('latitude', 'min'),
    lat_max=('latitude', 'max'),
    lon_min=('longitude', 'min'),
    lon_max=('longitude', 'max'),
    dist_sum_m=('dist_m', 'sum'),
).reset_index()
vehicle_daily['duration_hr'] = (vehicle_daily['end_time'] - vehicle_daily['start_time']).dt.total_seconds() / 3600
vehicle_daily.to_csv(PROCESSED_DATA_DIR / 'vehicle_daily_trajectory_summary.csv', index=False)
vehicle_daily.head()

In [ ]:
assignments['vehicle_id'] = pd.to_numeric(assignments['CarID'], errors='coerce').astype('Int64')
assignments = assignments.dropna(subset=['vehicle_id']).copy()
assignments['vehicle_id'] = assignments['vehicle_id'].astype(int)
assigned_ids = set(assignments['vehicle_id'])
all_ids = set(gps['vehicle_id'].unique())
unassigned_ids = sorted(all_ids - assigned_ids)
unassigned_ids

## 2. 停车热点、时长与未分配车辆

- 停车热点图用于观察常驻停留区域。
- 日里程热力图用于识别车辆在不同日期的活动强度。
- 对 101/104/105/106/107 重点看夜间停留、重复访问和与员工车的时间重叠。

In [ ]:
stop_events['start_time'] = pd.to_datetime(stop_events['start_time'])
stop_events['end_time'] = pd.to_datetime(stop_events['end_time'])
stop_events['start_hour'] = stop_events['start_time'].dt.hour
stop_events['is_night_stop'] = stop_events['start_hour'].between(0, 5)
stop_events['vehicle_group'] = np.where(stop_events['vehicle_id'].isin([101,104,105,106,107]), 'unassigned', 'assigned')
stop_vehicle_summary = stop_events.groupby(['vehicle_group', 'vehicle_id']).agg(
    stop_count=('duration_min', 'size'),
    total_stop_min=('duration_min', 'sum'),
    median_stop_min=('duration_min', 'median'),
    night_stop_count=('is_night_stop', 'sum'),
    avg_speed=('mean_speed_mps', 'mean'),
).reset_index().sort_values(['vehicle_group', 'stop_count'], ascending=[True, False])
stop_vehicle_summary.to_csv(PROCESSED_DATA_DIR / 'gps_stop_vehicle_summary.csv', index=False)

ua_stop = stop_events[stop_events['vehicle_id'].isin([101,104,105,106,107])].copy()
ua_hourly = ua_stop.groupby(['vehicle_id', 'start_hour']).size().reset_index(name='stop_count')
ua_hourly.to_csv(PROCESSED_DATA_DIR / 'unassigned_vehicle_stop_hourly.csv', index=False)
stop_vehicle_summary.head()

## 3. 交易复核、矛盾识别与时间精度控制

- CC 是分钟级，适合时间窗验证。
- loyalty 只有日期，只能做同日弱验证。
- `anomaly_transactions.csv` 作为优先复核清单，而不是最终结论。

In [ ]:
transactions['timestamp'] = pd.to_datetime(transactions['timestamp'])
transactions['date'] = pd.to_datetime(transactions['timestamp']).dt.date.astype('string')
cc = transactions[transactions['source'].eq('cc')].copy()
loyalty = transactions[transactions['source'].eq('loyalty')].copy()
cc['hour'] = pd.to_datetime(cc['timestamp']).dt.hour
cc['minute'] = pd.to_datetime(cc['timestamp']).dt.minute
cc['is_exact_noon'] = (cc['hour'].eq(12) & cc['minute'].eq(0))
cc['is_early_morning'] = cc['hour'].between(0,5)
cc_review = cc.merge(anomaly_transactions[['transaction_id','match_status','anomaly_reason']], on='transaction_id', how='left')
cc_review.to_csv(PROCESSED_DATA_DIR / 'q2_cc_review_table.csv', index=False)
cc_review[['transaction_id','timestamp','location_clean','price','card_id','match_status','anomaly_reason']].head()

In [ ]:
review = anomaly_transactions[['transaction_id','source','date','location_clean','price','card_id','anomaly_reason','match_status']].copy()
review['needs_gps_review'] = review['source'].eq('cc')
review.to_csv(PROCESSED_DATA_DIR / 'q2_contradiction_review.csv', index=False)
review.head()

## 4. 输出与图表

本节重绘并统一生成所有 Q2 图表，命名全部使用 `q2_*.png`。

In [ ]:
# Render / refresh Q2 figures
figdir = FIGURES_DIR
figdir.mkdir(parents=True, exist_ok=True)

# 1 stop hotspots
fig, ax = plt.subplots(figsize=(9,7))
ax.scatter(stop_events['mean_lon'], stop_events['mean_lat'], s=stop_events['duration_min'].clip(1,20)*6, alpha=0.35, color='#2a9d8f')
ax.set_title('Q2 GPS Stop Events Hotspots')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
fig.tight_layout(); fig.savefig(figdir / 'q2_1_stop_hotspots.png', dpi=180); plt.close(fig)

# 2 stop duration histogram
fig, ax = plt.subplots(figsize=(9,5))
sns.histplot(stop_events['duration_min'], bins=40, ax=ax, color='#264653')
ax.set_title('Q2 Stop Duration Distribution'); ax.set_xlabel('Duration (minutes)')
fig.tight_layout(); fig.savefig(figdir / 'q2_2_stop_duration_hist.png', dpi=180); plt.close(fig)

# 3 daily mileage heatmap
vehicle_daily['day'] = pd.to_datetime(vehicle_daily['date']).dt.day
piv = vehicle_daily.pivot_table(index='vehicle_id', columns='day', values='dist_sum_m', aggfunc='sum') / 1000
fig, ax = plt.subplots(figsize=(13,8))
sns.heatmap(piv, cmap='mako', ax=ax)
ax.set_title('Q2 Daily Mileage Heatmap by Vehicle (km)'); ax.set_xlabel('Day of January 2014'); ax.set_ylabel('Vehicle ID')
fig.tight_layout(); fig.savefig(figdir / 'q2_3_daily_mileage_heatmap.png', dpi=180); plt.close(fig)

# 4 unassigned vehicle distance boxplot
ua_daily = vehicle_daily[vehicle_daily['vehicle_id'].isin([101,104,105,106,107])].copy()
fig, ax = plt.subplots(figsize=(10,6))
sns.boxplot(data=ua_daily, x='vehicle_id', y='dist_sum_m', ax=ax, color='#e9c46a')
ax.set_title('Q2 Unassigned Vehicle Daily Distance'); ax.set_xlabel('Vehicle ID'); ax.set_ylabel('Distance (m)')
fig.tight_layout(); fig.savefig(figdir / 'q2_4_unassigned_distance_boxplot.png', dpi=180); plt.close(fig)

# 5 unassigned stop hours
fig, ax = plt.subplots(figsize=(10,6))
sns.histplot(ua_stop['start_hour'], bins=24, discrete=True, ax=ax, color='#e76f51')
ax.set_title('Q2 Unassigned Vehicle Stop Start Hours'); ax.set_xlabel('Hour of Day')
fig.tight_layout(); fig.savefig(figdir / 'q2_5_unassigned_stop_hours.png', dpi=180); plt.close(fig)

# 6 location-source counts
loc = transactions.groupby(['location_clean', 'source']).size().reset_index(name='count')
loc_top = loc.groupby('location_clean')['count'].sum().sort_values(ascending=False).head(12).index
fig, ax = plt.subplots(figsize=(12,6))
sns.barplot(data=loc[loc['location_clean'].isin(loc_top)], x='location_clean', y='count', hue='source', ax=ax)
ax.set_title('Q2 Transaction Counts by Location and Source'); ax.set_xlabel('Location'); ax.set_ylabel('Count'); ax.tick_params(axis='x', rotation=45)
fig.tight_layout(); fig.savefig(figdir / 'q2_6_location_source_counts.png', dpi=180); plt.close(fig)

# 7 anomaly price scatter
anom = anomaly_transactions.copy()
fig, ax = plt.subplots(figsize=(10,6))
sns.scatterplot(data=anom, x='date', y='price', hue='source', ax=ax)
ax.set_title('Q2 Anomaly Transaction Prices Over Time'); ax.set_xlabel('Date'); ax.set_ylabel('Price'); ax.tick_params(axis='x', rotation=45)
fig.tight_layout(); fig.savefig(figdir / 'q2_7_anomaly_price_scatter.png', dpi=180); plt.close(fig)

# 8 cc-loyalty match types
fig, ax = plt.subplots(figsize=(10,5))
sns.countplot(data=matched, x='match_type', order=matched['match_type'].value_counts().index, ax=ax, color='#457b9d')
ax.set_title('Q2 CC-Loyalty High-Confidence Match Types'); ax.set_xlabel('Match Type'); ax.set_ylabel('Count'); ax.tick_params(axis='x', rotation=45)
fig.tight_layout(); fig.savefig(figdir / 'q2_8_match_type_counts.png', dpi=180); plt.close(fig)

# 9 total stop duration by vehicle
fig, ax = plt.subplots(figsize=(12,6))
top_stop = stop_events.groupby('vehicle_id')['duration_min'].sum().sort_values(ascending=False).head(15).reset_index()
sns.barplot(data=top_stop, x='vehicle_id', y='duration_min', ax=ax, color='#2a9d8f')
ax.set_title('Q2 Total Stop Duration by Vehicle'); ax.set_xlabel('Vehicle ID'); ax.set_ylabel('Total Stop Minutes')
fig.tight_layout(); fig.savefig(figdir / 'q2_9_total_stop_duration_by_vehicle（附加图）.png', dpi=180); plt.close(fig)

# 10 stop count by day
fig, ax = plt.subplots(figsize=(12,5))
stop_day = stop_events.groupby(stop_events['start_time'].dt.date).size().reset_index(name='stop_count')
stop_day.columns = ['date','stop_count']
sns.lineplot(data=stop_day, x='date', y='stop_count', marker='o', ax=ax, color='#264653')
ax.set_title('Q2 Stop Count by Day'); ax.set_xlabel('Date'); ax.set_ylabel('Stop Count'); ax.tick_params(axis='x', rotation=45)
fig.tight_layout(); fig.savefig(figdir / 'q2_10_stop_count_by_day（附加图）.png', dpi=180); plt.close(fig)

# 11 daily transaction volume
trans_daily = transactions.groupby([transactions['timestamp'].dt.date, 'source']).size().reset_index(name='count')
trans_daily.columns = ['date','source','count']
fig, ax = plt.subplots(figsize=(12,5))
sns.lineplot(data=trans_daily, x='date', y='count', hue='source', marker='o', ax=ax)
ax.set_title('Q2 Daily Transaction Volume by Source'); ax.set_xlabel('Date'); ax.set_ylabel('Transactions'); ax.tick_params(axis='x', rotation=45)
fig.tight_layout(); fig.savefig(figdir / 'q2_11_daily_transaction_volume（附加图）.png', dpi=180); plt.close(fig)

# 12 top anomaly reasons
reason_counts = anom['anomaly_reason'].fillna('').str.split(';').explode().value_counts().head(12).reset_index()
reason_counts.columns = ['reason','count']
fig, ax = plt.subplots(figsize=(10,6))
sns.barplot(data=reason_counts, y='reason', x='count', ax=ax, color='#e76f51')
ax.set_title('Q2 Top Anomaly Reasons'); ax.set_xlabel('Count'); ax.set_ylabel('Reason')
fig.tight_layout(); fig.savefig(figdir / 'q2_12_top_anomaly_reasons（附加图）.png', dpi=180); plt.close(fig)

# 13 unassigned vehicle timeline
fig, ax = plt.subplots(figsize=(12,6))
sns.scatterplot(data=ua_stop, x='start_time', y='vehicle_id', size='duration_min', hue='vehicle_id', ax=ax, legend=False, sizes=(20,200))
ax.set_title('Q2 Unassigned Vehicle Stop Timeline'); ax.set_xlabel('Start Time'); ax.set_ylabel('Vehicle ID')
fig.tight_layout(); fig.savefig(figdir / 'q2_13_unassigned_vehicle_timeline（附加图）.png', dpi=180); plt.close(fig)

# 14 location category counts
loc_cat = transactions.groupby('location_category').size().sort_values(ascending=False).reset_index(name='count')
fig, ax = plt.subplots(figsize=(10,6))
sns.barplot(data=loc_cat, x='location_category', y='count', ax=ax, color='#457b9d')
ax.set_title('Q2 Transaction Counts by Location Category'); ax.set_xlabel('Location Category'); ax.set_ylabel('Transactions'); ax.tick_params(axis='x', rotation=45)
fig.tight_layout(); fig.savefig(figdir / 'q2_14_location_category_counts（附加图）.png', dpi=180); plt.close(fig)

sorted([p.name for p in figdir.glob('q2_*.png')])

## 5. 结论

### 关键结论
- GPS 停车事件已成功提取。
- 5 辆未分配车辆（101/104/105/106/107）已经进入独立分析层。
- `cc_loyalty_matched.csv` 仅表示高置信候选，不等同于最终身份结论。
- `loyalty` 因缺乏时间粒度，只能做日级弱验证。
- 若存在 GPS 与交易不一致，应优先视为需要进一步复核的矛盾，而不是直接当作真实异常。

### 给后续成员的可复用输出
- `gps_stop_events.csv`
- `vehicle_daily_trajectory_summary.csv`
- `gps_stop_vehicle_summary.csv`
- `unassigned_vehicle_daily_summary.csv`
- `unassigned_vehicle_stop_hourly.csv`
- `q2_cc_review_table.csv`
- `q2_contradiction_review.csv`

### 仍需注意的限制
- 交易地点没有在本任务中被显式重建为空间坐标，因此空间匹配仍以 GPS 停留与交易时段交叉验证为主。
- 如果后续获得交易地点坐标，可直接把本 notebook 的停车事件层与地点几何层做 spatial join。

In [ ]:
summary = {
    'stop_events': len(stop_events),
    'vehicle_daily_rows': len(vehicle_daily),
    'unassigned_vehicle_ids': unassigned_ids,
    'q2_figures': len(list(FIGURES_DIR.glob('q2_*.png'))),
    'high_conf_cc_loyalty_matches': len(matched),
    'review_transactions': len(anomaly_transactions),
}
summary

In [ ]:
summary_path = REPORTS_DIR / 'b_member_summary.md'
summary_path.write_text(f'''# Member B Summary

- Stop events extracted: {len(stop_events):,}
- Vehicle-day trajectory rows: {len(vehicle_daily):,}
- Unassigned vehicle IDs: {unassigned_ids}
- Q2 figures produced: {len(list(FIGURES_DIR.glob('q2_*.png'))):,}
- High-confidence CC-loyalty matches: {len(matched):,}
- Transactions flagged for review: {len(anomaly_transactions):,}

## Final interpretation

The GPS-derived stop-event layer is now usable for downstream card ownership, network, and suspicious-activity analysis. CC transactions can be time-checked against vehicle presence at minute precision, while loyalty transactions must remain day-level evidence only. Any remaining contradictions should be treated as investigation leads rather than final truth.
''', encoding='utf-8')
summary_path